#### Project 3: Autonomous Media Inclusivity Research Agent
## Day 1: Research, Scope and Architecture Planning

**Organisation:** Media Diversity Watch  
**Industry:** Media, Journalism and Publishing  
**Focus:** Intersectional inclusivity in media coverage and newsroom representation  
**Student:** Nova Track  
**Date:** 2026

---

## Project Introduction

### Use Case

```
Organisation: Media Diversity Watch
User: Nova Track, a media diversity researcher
Mission: Track whether major news outlets are getting better
         or worse at inclusive coverage over time, and publish
         that evidence to create public accountability.

Every Monday at 6am the N8N workflow triggers automatically.
The agent researches all 4 outlets: Al Jazeera English, The Guardian,
CNN, and New York Times.
By 9am Nova Track opens Notion and four scored, sourced reports are waiting.
She tracks scores week over week.
When an outlet's score drops significantly she has documented
evidence to support public advocacy.
No manual research. No guesswork. Evidence-based accountability.
```

### Project Statement

This project builds an autonomous AI agent for Media Diversity Watch,
a fictional media accountability NGO. The agent monitors four major
news outlets every week and evaluates them through an intersectional
inclusivity lens.

The user is Nova Track, a media diversity researcher who needs weekly
evidence-based reports on how Al Jazeera English, The Guardian, CNN,
and New York Times cover marginalized communities. Instead of manually
searching dozens of sources, the agent runs automatically every Monday
at 6am and saves four scored reports to Notion before she starts work.

The core belief: media shapes public perception. Consistent
underrepresentation or harmful framing of marginalized groups reinforces
real-world inequality. This agent tracks exactly that -- and builds a
historical record that can drive public accountability.

---

## Outlet Selection

The four outlets represent distinct editorial traditions and geographic perspectives:
Al Jazeera English (Global South broadcaster), The Guardian (UK progressive broadsheet),
CNN (US cable network), and New York Times (US newspaper of record).

RSS byline availability varies by outlet — The Guardian and NYT expose named journalist
bylines; Al Jazeera English and CNN do not. This absence is itself recorded as a
transparency signal under Angle 1, not treated as a data gap.

---

## The Four Evaluation Angles

For each community, the agent evaluates coverage across four angles:

**Angle 1: Representation in Bylines and Story Selection**
Are people from marginalised communities actually writing stories?
Are those stories getting published, platformed, and amplified?

**Angle 2: Portrayal Within Content**
When marginalised communities appear in coverage, what roles do they occupy?
Are they shown as experts, leaders, innovators, and everyday people --
or reduced to a single narrative of victim, criminal, or side character?

**Angle 3: Sourcing Diversity**
When journalists need an authority or expert quote, who do they call?
Are voices from within these communities treated as credible sources --
or does the outlet always default to outside perspectives speaking
about the community rather than from it?

**Angle 4: Language and Framing**
Does the outlet use inclusive, respectful, and accurate language?
Does framing center the humanity of these communities or does it
reduce them to a problem, a statistic, or a political debate?

---

## Communities Tracked

- Women and gender equality
- LGBTQ+ communities
- Racial and ethnic minorities
- People with disabilities

---

## What Makes This Agent Reliable

This agent is built to avoid hallucination -- the risk of an AI inventing
facts or scores. Every claim in the generated report is grounded in:

1. Retrieved documents from our Pinecone knowledge base (RAG)
2. Live articles fetched from NewsAPI
3. RSS feed data fetched directly from each outlet (primary source)
4. Wikipedia background data on each company

If the agent does not have evidence for a claim, it says so explicitly
rather than guessing. Every finding is tagged with its source.

---

## Known Limitations

- **RSS depth:** The RSS tool scans headlines and short teasers (1-3 sentences),
  not full article text. Keyword hit counts reflect surface-level signal only.
- **RAG scope:** The Pinecone knowledge base contains industry benchmarks and
  inclusivity research standards — it is the measuring stick, not outlet-specific
  evidence. It contextualises scores but cannot prove what a specific outlet does.
- **Full article access:** Fetching full article bodies would require scraping or
  paid API access (Guardian API, NYT API). This is a documented next step, not
  a project failure.

These constraints are enforced in the report generation prompt so the agent
acknowledges uncertainty rather than filling gaps with assumptions.

---

## Module 3 Skills Used

| Skill | Role in This Project |
|-------|---------------------|
| RAG + Pinecone | Knowledge base of verified inclusivity research (19 benchmark chunks) |
| LangChain | `@tool` decorator wraps each API into an agent-callable function |
| LangGraph | `create_react_agent` — ReAct loop: Thought → Action → Observation, repeated until research is complete |
| FastAPI | HTTP server exposing `/research` endpoint for N8N to call |
| N8N | Scheduled trigger (Mon 6am) and Notion report delivery |
| Prompt Engineering | Structured prompts with explicit anti-hallucination rules and scoring rubric |

## Project Architecture\n\nHow the agent works end to end -- from trigger to report.\n\n> **Architectural note:** Day 1 planned a fixed 5-node LangGraph graph. During implementation (Day 3),\n> the **ReAct (Reason + Act) pattern** was chosen instead. ReAct lets the agent decide dynamically\n> which tools to call and how many times, rather than following a hardcoded sequence.\n> This produces better-grounded research and requires significantly less boilerplate code.\n\n```\nUser Input / N8N Schedule Trigger\n        |\n        v\n[FastAPI Server — POST /research]\n        |\n        v\n[LangGraph ReAct Agent]\n    Thought: what do I need to know?\n    |\n    |-- Action: analyse_rss_feed(company)\n    |   Observation: bylines, coverage counts, language flags\n    |\n    |-- Action: rag_query_tool(query, angle) ×4\n    |   Observation: industry benchmarks from Pinecone knowledge base\n    |\n    |-- Action: search_newsapi(query)\n    |   Observation: what critics say about this outlet's diversity record\n    |\n    |-- Action: search_wikipedia(company)\n    |   Observation: ownership, editorial history, reach\n    |\n    Thought: I have enough evidence. Writing research summary...\n        |\n        v\n[Report Generator — Anthropic claude-sonnet-4-6]\n    Formats research into 11-section Markdown report\n        |\n        v\n[N8N Notion Node]\n        |\n        v\nNew Notion page in \"Media Inclusivity Reports\" database\n```\n\n**4 research tools the agent uses:**\n- Tool 1: `analyse_rss_feed` — primary source: today's bylines and language (no API key)\n- Tool 2: `rag_query_tool` — Pinecone RAG: curated inclusivity benchmarks\n- Tool 3: `search_newsapi` — live news: what journalists say about this outlet\n- Tool 4: `search_wikipedia` — company background (free REST API)\n\n**Output stored in two places:**\n- Local: `reports/{company_name}_{date}.md`\n- Remote: Notion page in the Media Inclusivity Reports database\n\n---

In [1]:
# CELL 1: Load environment variables and test our API connections
# This confirms all our API keys are working before we build anything

import os
from dotenv import load_dotenv

# Ensure working directory is project root (works whether run from root or notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load the .env file - this reads our secret API keys
load_dotenv()

# Check that each key was loaded (we print only the first 8 characters for safety)
keys_to_check = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "NEWSAPI_KEY": os.getenv("NEWSAPI_KEY"),
    "PINECONE_API_KEY": os.getenv("PINECONE_API_KEY"),
}

print("API Key Status Check:")
print("-" * 40)
for key_name, key_value in keys_to_check.items():
    if key_value:
        print(f"{key_name}: LOADED ({key_value[:8]}...)")
    else:
        print(f"{key_name}: MISSING - check your .env file!")

API Key Status Check:
----------------------------------------
OPENAI_API_KEY: LOADED (sk-svcac...)
ANTHROPIC_API_KEY: LOADED (sk-ant-a...)
NEWSAPI_KEY: LOADED (6ece1ad5...)
PINECONE_API_KEY: LOADED (pcsk_32Z...)


In [2]:
# CELL 2: Test NewsAPI - this is one of our 3 required real APIs
# NewsAPI lets us search for real news articles about any topic

from newsapi import NewsApiClient

# Initialize the NewsAPI client with our key
newsapi = NewsApiClient(api_key=os.getenv("NEWSAPI_KEY"))

# Test search: look for articles about gender representation in media
# This is directly related to our inclusivity focus
test_results = newsapi.get_everything(
    q="gender representation media journalism",
    language="en",
    sort_by="relevancy",
    page_size=3  # Just get 3 articles for testing
)

print(f"NewsAPI Connection: SUCCESS")
print(f"Total articles found: {test_results['totalResults']}")
print("\nSample article titles:")
print("-" * 40)
for article in test_results["articles"]:
    print(f"- {article['title']}")
    print(f"  Source: {article['source']['name']}")
    print()

NewsAPI Connection: SUCCESS
Total articles found: 7

Sample article titles:
----------------------------------------
- Katharine Burr Blodgett’s brilliance had to fit into the role of the only woman in a lab filled with men—it was the air she breathed
  Source: Scientific American

- What is the PC name for fisherman?
  Source: Lifesciencesworld.com

- Bostopia’s Evan George serves Boston daily news from a lefty perspective
  Source: Niemanlab.org



In [3]:
# CELL 3: Test OpenAI - we use this for embeddings (turning text into vectors)
# and as our main LLM for the agent's reasoning

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Quick test - ask something simple
test_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Say 'OpenAI connection successful' and nothing else."}
    ]
)

print(test_response.choices[0].message.content)

OpenAI connection successful.


In [4]:
# CELL 4: Test Pinecone - this is our vector database
# Think of it like a special database that stores meaning, not just text

from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# List existing indexes (should be empty on a fresh account)
existing_indexes = pc.list_indexes()
print("Pinecone Connection: SUCCESS")
print(f"Existing indexes: {existing_indexes.names() if existing_indexes else 'None yet'}")


Pinecone Connection: SUCCESS
Existing indexes: ['reranker-lab', 'trustworthy-ai-rag', 'media-inclusivity-index', 'n8n', 'rag-openai-index']


In [ ]:
# CELL 5: Research scope mapped to the four evaluation angles
# Every search query the agent runs comes from this structure

SAMPLE_COMPANIES = [
    "Al Jazeera English",
    "The Guardian",
    "CNN",
    "New York Times",
]

# Structure: angle -> community -> search queries
# This is what the agent actually searches for when researching a company
RESEARCH_DIMENSIONS = {

    # ANGLE 1: REPRESENTATION IN BYLINES AND STORY SELECTION
    "bylines_and_story_selection": {
        "women": [
            "women journalists bylines percentage",
            "female reporters front page stories",
            "women editors story selection media"
        ],
        "lgbtq": [
            "LGBTQ journalists newsroom bylines",
            "queer reporters story assignment media",
            "LGBTQ voices editorial decisions"
        ],
        "race": [
            "journalists of color bylines coverage",
            "Black Latino Asian reporters story selection",
            "racial diversity newsroom bylines"
        ],
        "disability": [
            "disabled journalists bylines media",
            "reporters with disabilities story coverage",
            "disability representation newsroom staff"
        ]
    },

    # ANGLE 2: PORTRAYAL WITHIN CONTENT
    "portrayal_in_content": {
        "women": [
            "women portrayed experts leaders media coverage",
            "female sources authority roles journalism",
            "victim framing women news coverage"
        ],
        "lgbtq": [
            "LGBTQ people portrayed beyond victims media",
            "transgender coverage roles representation",
            "queer community full portrayal journalism"
        ],
        "race": [
            "minorities portrayed criminals media coverage",
            "racial stereotyping news framing",
            "Black Latino community leaders coverage media"
        ],
        "disability": [
            "disability inspiration narrative media",
            "disabled people portrayed beyond victims journalism",
            "disability coverage framing stereotypes"
        ]
    },

    # ANGLE 3: SOURCING DIVERSITY
    "sourcing_diversity": {
        "women": [
            "women experts quoted news sources",
            "female authority sources journalism",
            "gender balance expert sources media"
        ],
        "lgbtq": [
            "LGBTQ experts sources quoted journalism",
            "queer voices authority media coverage",
            "LGBTQ community sources vs outside perspectives"
        ],
        "race": [
            "journalists of color expert sources quoted",
            "racial minority voices authority journalism",
            "diverse sourcing practices newsroom"
        ],
        "disability": [
            "disabled experts sources quoted media",
            "disability community voices journalism",
            "nothing about us without us media sourcing"
        ]
    },

    # ANGLE 4: LANGUAGE AND FRAMING
    "language_and_framing": {
        "women": [
            "gender neutral language media style guide",
            "sexist language journalism coverage",
            "inclusive language women reporting"
        ],
        "lgbtq": [
            "trans inclusive language media guidelines",
            "LGBTQ terminology accuracy journalism",
            "non-binary language news coverage"
        ],
        "race": [
            "racial language guidelines media",
            "dehumanizing language migrants refugees media",
            "race framing news coverage harmful language"
        ],
        "disability": [
            "disability language guidelines journalism",
            "ableist language media coverage",
            "person first identity first language news"
        ]
    }
}

# Report structure mapped to the four angles
REPORT_STRUCTURE = {
    "sections": [
        "1. Company Overview",
        "2. Angle 1: Representation in Bylines and Story Selection",
        "3. Angle 2: Portrayal Within Content",
        "4. Angle 3: Sourcing Diversity",
        "5. Angle 4: Language and Framing",
        "6. Community-by-Community Findings",
        "7. Key Strengths",
        "8. Areas of Concern and Harm Flags",
        "9. Inclusivity Score (1-10 per community, overall average)",
        "10. Recommendations",
        "11. Sources and Evidence"
    ]
}

print("Research dimensions defined across 4 angles and 4 communities.")
print(f"\nMonitored outlets: {SAMPLE_COMPANIES}")
print(f"\nAngles: {list(RESEARCH_DIMENSIONS.keys())}")
print(f"\nReport sections: {len(REPORT_STRUCTURE['sections'])}")

In [ ]:
# CELL 6: Save project configuration
# Reflects the updated four-angle structure

import json

project_config = {
    "project_name": "Autonomous Media Inclusivity Research Agent",
    "organisation": "Media Diversity Watch",
    "user_persona": "Nova Track, media diversity researcher",
    "use_case": "Weekly automated monitoring of 4 major outlets with Notion delivery",
    "schedule": "Monday 6am automatic, or on-demand via form trigger",
    "notion_integration": True,
    "rss_tool_enabled": True,
    "industry": "Media, Journalism and Publishing",
    "focus": "Intersectional inclusivity in media coverage and newsroom representation",
    "communities_tracked": [
        "women",
        "lgbtq",
        "race",
        "disability"
    ],
    "evaluation_angles": [
        "bylines_and_story_selection",
        "portrayal_in_content",
        "sourcing_diversity",
        "language_and_framing"
    ],
    "sample_companies": SAMPLE_COMPANIES,
    "research_dimensions": RESEARCH_DIMENSIONS,
    "report_structure": REPORT_STRUCTURE,
    "scoring_method": "equal_weight_with_harm_flags",
    "pinecone_index_name": "media-inclusivity-index",
    "embedding_model": "text-embedding-ada-002",
    "llm_model": "gpt-4o-mini",
    "chunk_size": 500,
    "chunk_overlap": 50
}

config_path = "data/project_config.json"
with open(config_path, "w") as f:
    json.dump(project_config, f, indent=2)

print(f"Project configuration saved to {config_path}")
print(f"\nOrganisation: {project_config['organisation']}")
print(f"User: {project_config['user_persona']}")
print(f"Schedule: {project_config['schedule']}")
print(f"Outlets: {project_config['sample_companies']}")
print(f"Evaluation angles: {project_config['evaluation_angles']}")
print(f"Communities tracked: {project_config['communities_tracked']}")
print(f"Scoring method: {project_config['scoring_method']}")

#### Day 1 Complete - Summary

##### What I did today:
- Set up project folder structure
- Installed all required packages
- Verified all 4 API connections work
- Defined our research scope and focus
- Documented project architecture
- Saved project configuration

##### What I am building:
An agent that autonomously researches media companies and generates
reports on their intersectional inclusivity across gender, LGBTQ+, race, and disability: no human is needed once it is triggered.

##### Next up - Day 2:
- Create Pinecone vector index
- Collect and process documents about media inclusivity
- Build the RAG retrieval pipeline